In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from google.genai import types

# 1. Initialize environment variables and clients
load_dotenv(override=True)

openai_client = OpenAI()       # Uses OPENAI_API_KEY
anthropic_client = Anthropic() # Uses ANTHROPIC_API_KEY
gemini_client = genai.Client() # Uses GEMINI_API_KEY

# 2. Define Model Names (Using efficient versions for orchestration testing)
GPT_MODEL = "gpt-5-mini"
CLAUDE_MODEL = "claude-haiku-4-5"
GEMINI_MODEL = "gemini-2.5-flash"

# 3. Define Persona System Prompts
alex_system = """
You are Alex, a chatbot who is very argumentative. 
You disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a 3-way conversation with Blake and Charlie.
Keep your response concise (under 3 sentences).
"""

blake_system = """
You are Blake, a chatbot who is very polite. 
try to agree with everything or find common ground. If others are argumentative, you try to calm them down.
You are in a 3-way conversation with Alex and Charlie.
Keep your response concise (under 3 sentences).
"""

charlie_system = """
You are Charlie, a chatbot who is completely objective and data-driven.
You look at things from a purely logical perspective, pointing out flaws in both emotional agreements and stubborn arguments.
You are in a 3-way conversation with Alex and Blake.
Keep your response concise (under 3 sentences).
"""

# 4. Centralized Stateless Conversation History 
conversation_history = []

def get_formatted_conversation():
    """Helper function to compile conversation history into a custom template string."""
    return "\n".join(conversation_history)

# Agent Callers
def call_alex(conversation):
    messages = [
        {"role": "system", "content": alex_system},
        {"role": "user", "content": f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""}
    ]
    response = openai_client.chat.completions.create(model=GPT_MODEL, messages=messages)
    return response.choices[0].message.content.strip()

def call_blake(conversation):
    # Anthropic native SDK call with explicit system parameter
    message = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=800,
        system=blake_system,
        messages=[
            {"role": "user", "content": f"""
You are Blake, in conversation with Alex and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Blake.
"""}
        ]
    )
    return message.content[0].text.strip()

def call_charlie(conversation):
    config = types.GenerateContentConfig(
        system_instruction=charlie_system,
        max_output_tokens=800
    )
    prompt = f"""
You are Charlie, in conversation with Alex and Blake.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Charlie.
"""
    
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=config
    )
    return response.text.strip()

# ─── Orchestration Loop ───
print("Starting 3-Model Conversation...\n")
print("-" * 60)

# Simulate 3 full rounds of 3-way conversation
for round_idx in range(1, 4):
    print(f"\n===== ROUND {round_idx} =====")

    # 1. Alex with GPT
    current_conversation = get_formatted_conversation()
    alex_reply = call_alex(current_conversation)
    print(f"\n[Alex]: {alex_reply}")
    conversation_history.append(f"Alex: {alex_reply}")

    # 2. Blake with Claude
    current_conversation = get_formatted_conversation()
    blake_reply = call_blake(current_conversation)
    print(f"\n[Blake]: {blake_reply}")
    conversation_history.append(f"Blake: {blake_reply}")
    
    # 3. Charlie with Gemini
    current_conversation = get_formatted_conversation()
    charlie_reply = call_charlie(current_conversation)
    print(f"\n[Charlie]: {charlie_reply}")
    conversation_history.append(f"Charlie: {charlie_reply}")